# 第３回

##  自己情報量・平均情報量

- サイコロを例に計算してみよう。均質なサイコロA：どの目が出る確率も $\frac{1}{6}$ と 偏ったサイコロB：1が出る確率 $\frac{1}{2}$ それ以外 $\frac{1}{10}$
- この時、自己情報量はそれぞれ
  - $I(A,1) = -\ln \frac{1}{6}$
  - $I(B,1) = -\ln \frac{1}{2}$
  - $I(B,2) = -\ln \frac{1}{10}$

In [2]:
import numpy as np

i_a_1 = - np.log(1/6)
i_b_1 = - np.log(1/2)
i_b_2 = - np.log(1/10)

In [15]:
print(f"{i_a_1:.3f}")
print(f"{i_b_1:.3f}")
print(f"{i_b_2:.3f}")


1.792
0.693
2.303


- 同様に平均情報量を求めてみよう
- $H= - \sum_{x \in X} P(x) \ln P(x) $

In [18]:
P_a = [1/6]*6
print(P_a)

[0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666, 0.16666666666666666]


In [19]:
H_a = np.sum([ -P_a[i]*np.log(P_a[i]) for i in range(0,len(P_a),1)])

In [25]:
print(f"{H_a:.3f}")

1.792


In [22]:
P_b = [1/2] + [1/10]*5
print(P_b)

[0.5, 0.1, 0.1, 0.1, 0.1, 0.1]


In [23]:
H_b = np.sum([ -P_b[i]*np.log(P_b[i]) for i in range(0,len(P_b),1)])

In [26]:
print(f"{H_b:.3f}")

1.498


##  xpathを使ったウェブスクレイピング

In [27]:
from lxml import html

with open("./c3.html", 'r') as f:
    fstr = f.read()

# lxmlでHTMLを解析
tree = html.fromstring(fstr)

# XPathでニュースタイトルを取得（GoogleChrome上の）

# どちらでも取れるのを確認
#titles = tree.xpath('/html/body/div/text()')
titles = tree.xpath('//div/text()')

# 結果を出力
for title in titles:
    print(title)


こんにちは世界



In [28]:
#response.content

## クローリング

### APIの例（郵便番号検索API）

In [29]:
import requests

In [30]:
res = requests.get("https://zipcloud.ibsnet.co.jp/api/search", params={'zipcode': '8701124'})

In [31]:
resDic = res.json()

In [32]:
resDic

{'message': None,
 'results': [{'address1': '大分県',
   'address2': '大分市',
   'address3': '旦野原',
   'kana1': 'ｵｵｲﾀｹﾝ',
   'kana2': 'ｵｵｲﾀｼ',
   'kana3': 'ﾀﾞﾝﾉﾊﾙ',
   'prefcode': '44',
   'zipcode': '8701124'}],
 'status': 200}

In [33]:
adDic1 = resDic["results"][0]

In [34]:
print(f"{adDic1['address1']}{adDic1['address2']}{adDic1['address3']}")

大分県大分市旦野原


### クローリング演習

#### 食べログ飲食店データ取得

##### 1. 取得したいレビューのURLの取得

In [35]:
baseurl = "https://tabelog.com"
URL=baseurl+"/oita/A4402/A440202/44005660/dtlrvwlst/"

In [36]:
URL

'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/'

In [37]:
res = requests.get(URL)

In [38]:
tree = html.fromstring(res.content)

In [39]:
eles = tree.xpath("//a[@class='rvw-item__title-target']")

In [40]:
reviewurls = []
for ele in eles:
    aurl = baseurl+ele.get("href")
    reviewurls.append(aurl)

In [41]:
set(reviewurls)

{'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B382464585/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B417580252/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B440585631/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B440868409/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B453057681/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B458471005/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B478849487/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B482850549/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B494269956/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B526938820/?use_type=0&smp=1',
 'https://tabelog.com/oita/A4402/A440202/44005660/dtlrvwlst/B84684293/?use_type=0&smp=1'}

#### 2. 各レビューから取得したい内容のみを取得
- userid, name, rating, title, review
- 1つのURLでやってみる

In [42]:
rurl = reviewurls[0]
res = requests.get(rurl)
tree = html.fromstring(res.content)

In [43]:
#res.content

In [44]:
# User id/ Name
eles = tree.xpath("//a[@class='rvw-item__rvwr-name']")

In [45]:
eles[0].get("href")

'/rvwr/025510967/'

In [46]:
eles[0].getchildren()[0].text

'べっぴぃえんちゃん'

In [47]:
# rating
eles = tree.xpath("//div[@class='rvw-item__rvw-info']//b")

In [48]:
eles[0].text

'4.0'

In [49]:
#title
eles = tree.xpath("//p[@class='rvw-item__title']/strong")

In [50]:
eles[0].text

'いつも新たな出会いのあるお店'

In [51]:
# review
#eles = tree.xpath("//div[@class='rvw-item__rvw-comment']")
#eles = tree.xpath("//div[contains(@class, 'rvw-item') and contains(@class, 'rvw-comment')]")
eles = tree.xpath("//div[contains(@class, 'rvw-comment')]/p")

In [52]:
eles

[<Element p at 0xffff3e59f520>, <Element p at 0xffff3e59ce60>]

In [53]:
rawreview = eles[0].xpath("string()")

In [54]:
rawreview

'\n          3ヶ月ぶりの『トムヤムクン。』。お伺いするのは5、6回目だと思うが、来る度に新しい料理との出会いがある。メニューがなく、その時々の旬のもの、美味しいものを出してくれるシェフのおかげだ。アペリティフからして超独創的。こんな冷麺、見たことも食べたこともない。アジ、サバ、アサリの出汁に、新生姜とガスパチョ風のキュウリ。これだけでも斬新だが、ホタテの醤油漬けと一緒に食べると、あら不思議。塩ラーメンが一転醤油ラーメンに。本当に初めての味わいだ。次はクロダイのサラダ。コリンキーやズッキーニ、トマトはピクルスに。そして水菜、パクチーなど他の野菜やクロダイと一緒にキウイフルーツのソースで食べる。こんな発想聞いたことがない。魚は、一皿めはヘダイ。揚げたヘダイにとうもろこしのソース。とうもろこしが甘い。二皿めはイトヨリダイ。1番下に新タマネギのシャリアピンソース。魚の上には細かく刻んだナスとつるむらさき。オレガノも載っている。おそらくこれが味の決め手だろう。最後になって漸くタイ料理らしくなってきた（笑）まず看板料理のプーパッポンカリー。〆はエビのレッドカレー。レアなエビが美味しい。個人的にはもう少し辛くても良いが。まだまだ食べ足りない気もするが、ワインも空いたことだし、今日はここまで。ご馳走さまでした。今度は7皿でお願いするぞ〜\n        '

In [55]:
review = rawreview.strip()

In [56]:
# ここまでの内容を関数化
def scraper(rurl):
    res = requests.get(rurl)
    tree = html.fromstring(res.content)
    # User id/ Name
    eles = tree.xpath("//a[@class='rvw-item__rvwr-name']")
    uid = eles[0].get("href")
    name = eles[0].getchildren()[0].text
    # rating
    eles = tree.xpath("//div[@class='rvw-item__rvw-info']//b")
    rating = eles[0].text
    #title
    eles = tree.xpath("//p[@class='rvw-item__title']/strong")
    title = eles[0].text
    # review
    #eles = tree.xpath("//div[@class='rvw-item__rvw-comment']")
    #eles = tree.xpath("//div[contains(@class, 'rvw-item') and contains(@class, 'rvw-comment')]")
    eles = tree.xpath("//div[contains(@class, 'rvw-comment')]/p")
    rawreview = eles[0].xpath("string()")
    review = rawreview.strip()
    
    return uid, name, rating, title, review

In [57]:
rows = [["uid","name","rating", "title", "review"]]
for rurl in list(set(reviewurls)):
    row = scraper(rurl)
    rows.append(row)

In [58]:
import pandas as pd
pd.options.display.max_colwidth = 250

df = pd.DataFrame(rows[1:],columns=rows[0])

In [59]:
df

,uid,name,rating,title,review
0,/rvwr/010815537/,ALUCARD,4.5,今月２度目です,今夜は７皿の料理でオーダー昼からウイスキー空にしてたのに元気よかった＾＾
1,/rvwr/010551291/,パパイヤ石塚,4.0,Theイノベーティブ,ご近所の猫旅館が素泊まりのみのため土曜日の晩御飯に夫婦で伺いました。昼夜限らず1日1組限定との事でしたが1週間少し前で運良く予約が取れました。他のコメントにもあるようににいつ予約の電話をしたら良いか迷いながらスマホで夕方頃に入電。繋がりませんでしたがSMSでお返事をいただき、SMSのやり取りで予約が完結できました。移動の関係で到着が遅れてしまいましたが、快く対応いただけました。（電話がギリギリまでつながらず焦りましたが）お店は商店街の一角にありました。開店後、十数年経つとの事でしたが店内...
2,/rvwr/polpol/,polpol,4.5,別府でタイヌーベルキューイジーヌ、素晴らしい品々を,￼マイル消化で夫婦1泊2日の大分旅行を決め、別府温泉へ￼別府温泉には約50年ぶりの訪問。宿泊は温泉旅館￼だったので、晩御飯は旅館にて。と言うわけで、お昼に何かおいしいものを食べようと検索、￼こちらのお店に巡り会いました。大分の郷土料理も良いのですが、タイ料理をしばらく食べてなかったなぁと思ったので。レビューを読むと、シェフお1人で切り盛りされているとの事。￼先月1日1組の向ヶ丘遊園のイタリアンに行ったばかり、￼その魅力（シェフの考える世界の具現化）を知ってしまったので、とても興味を持ち予...
3,/rvwr/005332034/,フラワ田,3.9,サスティナブルなスタイル,・冷麺 海老の頭や鰹、浅利などから出汁原木椎茸は含め煮に つるむらさき 木耳甘いトマトのピューレ・クログジ 3日熟成、あかもく野菜のコンビネーションサラダ新生姜、コリンキーのソース・ガパオライスコロッケ黄身ソース、海老のトマトソース・ヤマトカマス 玉蜀黍 ココナッツミルクのピューレ、里芋の揚げ餅・ソフトシェルクラブのプーパッポンカリー・2日マリネしたみつせ鶏のコンフィ青唐辛子のハリッサとミントのソース・グリーンカレー サザエの肝ソースのリゾット ゴールデンフェニックス新牛蒡とサザエ メバ...
4,/rvwr/obatablog/,③オバタ,5.0,大分県は別府市にやって来ました！✧\(>o<)ﾉ✧,グルメな御方のレビューを見て、ソッコーその御方にお願いして一緒に行ってもらいました✧\(>o<)ﾉ✧間を開けない再訪問にも快く同行して頂いてありがとうございます！ಥ‿ಥてか別府市って本当ディープですね〜海辺の街ではありますが、海鮮より多国籍料理の方が興味津々です⊙.☉ま、海鮮も行きたいところはあるんですがね(◍•ᴗ•◍)ここに来る前に、すでにコリアン立飲みで引っ掛けてたので軽くほろ酔い気味ですよಥ‿ಥコースは貸し切りなのでレスポンスよく始まります♪どれもレベルが高く、皿のバランスも素晴ら...
5,/rvwr/kumachelin/,クマシュラン,4.6,ヌーベルタイジーヌ,福岡の天ぷら屋の大将に勧められて訪問。一日に昼か夜の一組。これはタイ料理なのか？と驚く素晴らしい繊細な料理ばかり。シェフも楽しい方で、色々お話していただけた。旨くて楽しくて、また絶対に行きたいお店。
6,/rvwr/025510967/,べっぴぃえんちゃん,4.0,いつも新たな出会いのあるお店,3ヶ月ぶりの『トムヤムクン。』。お伺いするのは5、6回目だと思うが、来る度に新しい料理との出会いがある。メニューがなく、その時々の旬のもの、美味しいものを出してくれるシェフのおかげだ。アペリティフからして超独創的。こんな冷麺、見たことも食べたこともない。アジ、サバ、アサリの出汁に、新生姜とガスパチョ風のキュウリ。これだけでも斬新だが、ホタテの醤油漬けと一緒に食べると、あら不思議。塩ラーメンが一転醤油ラーメンに。本当に初めての味わいだ。次はクロダイのサラダ。コリンキーやズッキーニ、トマトは...
7,/rvwr/000617475/,☆中隊長☆,3.9,【泰上々颱風】都会よりも洗練されたモダンタイランドは湯の街に♪,2021年長月だいぶ昔に行った時とは、完全にスタイルが変わってしまったコチラのお店！良い意味で洗練された料理を提供で【1日/1組のみの完全予約制】！？興味ワキワキだけどなかなか行く機会無かったが偶然の賜物で行けたよwwそして無茶ぶりの被害者がまた出た模様だけど！？（謎）【ディナーコースSPver(10000円）】※事前に食材をグレードアップ。〇長崎県産灸鰹の塩タタキと大分県産地野菜のピクルス人参のドレッシング様〇大分和牛もも肉のラープヌア大分県産雲丹乗せ様〇高知県産ノドグロの白ワイン蒸し...
8,/rvwr/000057392/,fujimo123,4.0,湯の街に佇む本場以上にハイレベルなタイ料理,理別府に来てます。今宵ディナーはこちらで予約。地元人でも有名な浜脇温泉地区に位置する料理タイ料理専門店。オーナーが1人で切り盛りしているため1日昼・夜各１組限定の完全予約制なので、今では2～3か月先しか取れないほど人気。メニューはお任せコースのみで、料金は品数で変わるシステム。再訪で今回も5品（４０００円）と、スパークリングワイン銘柄お任せでフルボトル１本を４人でシェアー。（計５５００円強）店内は１卓のみのこじんまりも相変わらず。まずはスパークリングワインで喉を潤してからコーススタート。...
9,/rvwr/003971849/,しばきち,3.8,別府のタイ料理店。日本人向けのタイ料理を堪能出来て満足。,大分県別府市浜脇にあるタイ料理専門店。以前から行きたかったお店でようやく初訪問。完全予約制との事だったので事前に予約してからの訪問。店主は日本人。別府は多国籍な料理が多くその国の人が経営してることが多い。なので、こういうお店で日本人の店主というのは珍しく感じるね。色々なタイ料理を注文してシェア。中でも気に入ったのはガパオライスと海老の入ったタイ風焼飯。ガパオライスはナンプラーの風味も良くスパイシーな感じはあまりなくマイルド。挽肉はたっぷり。肉感が十分に感じられて美味しい。日本向けのガパオ...


### BlueSkyデータの取得（API）

#### APIから投稿してみる

In [60]:
from atproto import Client
import pandas as pd

In [61]:
import json

with open("bsky_account.json", 'r') as f:
    acdic = json.load(f)

In [62]:
client = Client(base_url='https://bsky.social')
client.login(acdic["account"], acdic["password"])

ProfileViewDetailed(did='did:plc:7sqtnkepsjplcvuqpvprsx52', handle='bcoredsclass0001.bsky.social', associated=ProfileAssociated(chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated', activitySubscription={'allowSubscriptions': 'followers'}), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:7sqtnkepsjplcvuqpvprsx52/bafkreif4w23vqx2keibm2z665tm4i77kewnamclqtompktx6z6acdl2jdq', banner=None, created_at='2025-04-10T00:59:16.772Z', description=None, display_name='', followers_count=20, follows_count=1, indexed_at='2025-04-10T00:59:16.772Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=5, verification=None, viewer=ViewerState(blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=None, muted=False, muted_by_list=None, py_type='app.bsky.actor.defs#viewerState'), py_type='app.bsky.actor.defs#profileViewDetailed')

In [63]:
client.send_post("今日はB-Coreで授業です")

CreateRecordResponse(uri='at://did:plc:7sqtnkepsjplcvuqpvprsx52/app.bsky.feed.post/3mkhycq5v7k2e', cid='bafyreibu4vrwvcypnopyvjlcxbuxlfhysb3evqtonxeeqyrkn3e4czyclu')

#### キーワード検索

In [64]:
res = client.app.bsky.feed.search_posts({"q":"ミャクミャク", "sort":"top", "limit":100, "cursor":None})

In [65]:
print(res.cursor)
print(res.hits_total)

rows = [["name","text"]]
for post in res.posts:
#    print(post.cid)
#    print(post.author.did)
#    print(post.author.handle)
#    print(post.author.display_name)
#    print(post.author.avatar)
#    print(post.record.text)
#    print(post.like_count)
#    print(post.quote_count)
#    print(post.reply_count)
#    print(post.repost_count)
#    print(post.uri)
    name = post.author.display_name
    text = post.record.text.strip()
    row = [name, text]
    rows.append(row)

100
None


In [66]:
df = pd.DataFrame(rows[1:], columns=rows[0])

In [67]:
df

,name,text
0,ねこのしっぽ,OAZO丸善にミャクミャク（売り場）が復活していました\nついシール買っちゃった\nわーい、可愛い〜〜🥰\nところでどうしようこれ〜〜\n使い道がわからな〜〜い😆
1,ナシオナシ🖋️絵日記,🥢今夜は漬け丼〜オクラ、とろろ、納豆も盛ったよ！もっちりミャクミャク＆ノーマルちゃん\n🥢茄子焼いて「しょうゆ麹」と砂糖、お酢で絡めた🍆美味しい\n🥢煮物\n\n#青空ごはん部 #今日のごはん #おうちごはん #自炊班 #washoku\n#ぬい撮り
2,ナシオナシ🖋️絵日記,🔶ハンバーグ！柔らかくて優しい味になった\n🔶鮭の粒マスタードはちみつ照り焼き\n🔶ブロッコリーをだし醤油、マヨにつけてから揚げ焼き🥦\n香ばしくなるかなと衣にナッツも加えてみた\n全部グリル（ココットプレート）で焼いたよ便利だね、もっちりミャクミャクちゃん\n\n#青空ごはん部 #今日のごはん #おうちごはん #自炊班\n#ぬい撮り
3,江東区民,夕飯は、ミャクミャクひっぱりたこ飯をいただきました！\n去年予約してたのが、ようやく届きました\n#青空ごはん部
4,ナシオナシ🖋️絵日記,「キツネとレモン」のオオカミ缶チョコレート可愛い🐺💕 （Morozoff）イチゴ、ハニーキャラメルなど6種のチョコが入ってました\n＼ﾜｰｲ！ﾁｮｺ食ﾍﾞﾀｲｰ!／\nな、もっちりミャクミャク＆推しピンクちゃん\n\n#甘党部 #chocolate\n#ぬい撮り
...,...,...
95,サーダ,ミャクミャク対太陽の塔\nべらぼうしょうぶだ！
96,びるまち,ミャクミャク様に脳を焼かれた民が関西は多いんじゃ・・・・\nミャクミャクのCMが流れると家族一同動きが止まりテレビを眺めてる\nx.com/expo2025prod...
97,ナシオナシ🖋️絵日記,🥢白菜と鶏肉のうま煮\n🥢ブリのネギ味噌焼き\n🥢はんぺんのたらこマヨがけ\n🥢味噌汁\n炊きたてご飯だよ〜もっちりミャクミャク＆フロッキーちゃん🍚\n\n#青空ごはん部 #今日のごはん #おうちごはん #自炊班\n#ぬい撮り #ぬい活
98,ヒラ🎨🦎🎮,ちゃんとツーショ収めてきましたよ


### ES基盤から検索してみる（研究室のサーバに負荷がかかるので注意）

In [68]:
from elasticsearch import Elasticsearch
import datetime as dt
from dateutil.relativedelta import relativedelta

In [69]:
# Elasticsearch の接続先
es = Elasticsearch("http://cicero.csis.oita-u.ac.jp:9200")

In [109]:
def get_post_uris(query, start, end):
    # クエリ条件に合致する投稿を取得（例では commit.record.text に対して match クエリ）
    body = {
        "size": 1000,  # 必要件数に応じて調整
        "timeout": "1m",
        "query": {
            "bool": {
                "must": [
                    {"match": {"commit.record.text": query}},
                    {"range": {"commit.record.createdAt": {"gte": start, "lte": end}}}
                ]
            }
        }
    }
    res =  es.options(request_timeout=60).search(index="postindex-*", body=body)
    posts = []  # 各投稿の情報（URI やテキストなど）
    uris = []   # 再構築した URI のリスト
    for hit in res["hits"]["hits"]:
        src = hit["_source"]
        try:
            did  = src["did"]
            coll = src["commit"]["collection"]
            rkey = src["commit"]["rkey"]
            # URI を "at://{did}/{collection}/{rkey}" として再構築
            uri = f"at://{did}/{coll}/{rkey}"
            uris.append(uri)
            # ここではテキスト（全文の一部など）も取得しておく（任意）
            text = src["commit"]["record"].get("text", "")
            posts.append({"uri": uri, "text": text})
        except Exception as e:
            continue
    return posts, uris

In [174]:
def get_like_ranking(query, start, end):
    posts, uris = get_post_uris(query, start, end)
    if not uris:
        return pd.DataFrame()

    body = {
        "size": 0,
        "timeout": "1m",
        "query": {
            "bool": {
                "filter": [
                    {"terms": {"commit.record.subject.uri.keyword": uris}},
                    {"term": {"commit.operation.keyword": "create"}},
                    {
                        "range": {
                            "commit.record.createdAt": {
                                "gte": start,
                                "lte": end
                            }
                        }
                    }
                ]
            }
        },
        "aggs": {
            "post_likes": {
                "terms": {
                    "field": "commit.record.subject.uri.keyword",
                    "size": len(uris)
                }
            }
        }
    }
    res = es.options(request_timeout=60).search(index="likeindex-*", body=body)

    buckets = res["aggregations"]["post_likes"]["buckets"]

    df_aggs = pd.DataFrame([{"uri": b["key"], "like_count": b["doc_count"]} for b in buckets])
    posts_df = pd.DataFrame(posts)
#    st.dataframe(posts_df)
    merged = pd.merge(posts_df, df_aggs, on="uri", how="left").fillna(0)
    merged["like_count"] = merged["like_count"].astype(int)
    merged = merged.sort_values("like_count", ascending=False)
    return merged

In [175]:
#today = dt.date.today()

In [176]:
#start_date = today + relativedelta(days=-2)
start_date = dt.date(2026,1,1)

In [177]:
start_date

datetime.date(2026, 1, 1)

In [178]:
start_date_str = start_date.strftime("%Y-%m-%dT00:00:00Z")

In [179]:
start_date_str

'2026-01-01T00:00:00Z'

In [180]:
end_date = dt.date(2026,1,2)

In [181]:
#end_date_str = today.strftime("%Y-%m-%dT23:59:59Z")
end_date_str = end_date.strftime("%Y-%m-%dT23:59:59Z")


In [182]:
posts, uris = get_post_uris(query="ミャクミャク", start=start_date_str, end=end_date_str)

In [183]:
df = get_like_ranking(query="ミャクミャク", start=start_date_str, end=end_date_str)

In [184]:
df

,uri,text,like_count
36,at://did:plc:vttjhehv2sasiplcktnz6vxt/app.bsky.feed.post/3mbdqjeiuwk2k,万博とミャクミャクがテレビに映るたびに「ところで業者への未払金払ったわけ？」「未払で訴えられてる」「着ぐるみの中に吉村ぶちこんで出稼ぎさせて支払わせるべき」「家財差し押さえろよ」の会話が実家内メンバーを変え延々擦られている,326
37,at://did:plc:a6gnkzxmgqgokl2ydlss5jwq/app.bsky.feed.post/3mbeev7z3p22n,"🥢おせち！\n我が家は加賀料理が愉しめる「大志満」からのお取り寄せです。美味しい(●´ㅂ`●)""ŧ‹”ŧ‹”\nピンクのミャクミャク達も一緒に…正月なのでもっふもふゴージャスになったもっちり\n\n#青空ごはん部 #今日のごはん #おうちごはん #ぬい撮り #新年",39
39,at://did:plc:a6gnkzxmgqgokl2ydlss5jwq/app.bsky.feed.post/3mbgv5skmik23,冷える日は鍋であったまろ！なミャクミャク達\n正月って事でちょっと良いお肉（我が家比）を買ったんで、友人からおススメされたヤマサの「うま肉鍋」の鍋つゆで！ダシがしっかりしてて美味しい\n\n#青空ごはん部 #今日のごはん #おうちごはん #ぬい撮り,23
12,at://did:plc:ybsksw7ink4rzcj2s6y2mbfy/app.bsky.feed.post/3mbdwsunfg22q,ミャクミャクちゃんと一緒に蟹食べてきたよん🥰\n#ぬい撮り\n#ぬいぐるみ部,18
28,at://did:plc:a6gnkzxmgqgokl2ydlss5jwq/app.bsky.feed.post/3mbebslbxd22n,🎍Happy new year！🐴✨\n新年あけましておめでとうございます\nもっちり＆日焼けミャクミャク達と迎える2026元日\n\n#ぬい撮り #ぬい活 #青空写真部 #stuffedanimal,15
24,at://did:plc:sj6rhyvfejbmmt5krdtdg6fn/app.bsky.feed.post/3mbdskh7gf22h,あけましておめでとうございます⛩\n和菓子をいただくぞ✨\n(さりげに混ざるミャクミャク様)\n#とうらぶぬい部\n#とうらぶおもち部\n#ぬい撮り,9
33,at://did:plc:456bfsxnvscqxjnsotaicbvz/app.bsky.feed.post/3mbgkboaxkc24,ミャクミャク5000円福袋開けました\nメインのミャクミャクぬいかわいかったのでよかったです。ぬい持ってないし\nこれだけでも定価高いので他大したもん入ってないやろと思ってましたが\nねんどろいどイコちゃん入っててびっくり（単体でもエキマルシェでセール価格になってたけど）\nあとNゲージさくらとかチョロＱかがやき\nN700系アクスタなど\nいろいろJRぽいいいものも入ってましたが\nミャクミャクバイザーは要らんかった……かわいいけど会場内のどこでかぶるのかw,8
34,at://did:plc:bu2e6276iftb53d6lgfwhufe/app.bsky.feed.post/3mbgp6ksz6223,デザイナーさんとかお好きな方々には申し訳ないんだけど、俺は「ミャクミャクと会える施設」ってフレーズで、精神病院みたいなもんが作られると、早合点してしまった。\n\nnews.yahoo.co.jp/pickup/6564342,5
4,at://did:plc:smyqzgc7tofvum4y4jfkz3zp/app.bsky.feed.post/3mbdlwkiszk2o,ばあば、ついにポンデリングの事をミャクミャクと言い出した,4
26,at://did:plc:t47zlebq4dzykumcn4fxremt/app.bsky.feed.post/3mbfibrqxkc2r,ミャクミャクはキュートだし国内にいて縁のない国の様々な物が見れるのって良いよなと思うけど費用未払い全く解決してなくて引いてしまう…………,4
